# SignInLogs Hello World

This notebook demonstrates connecting to Microsoft Sentinel Data Lake, retrieving data from some common tables, and constructing a simple notebook from those.

## Getting Started

Update the `WORKSPACE_NAME` line to match a workspace in your tenant which has `SignInLogs` and `EntraUsers` tables.

```python
# Configuration - Update this with your workspace name
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"

```

## Microsoft Documentation

- [Blog: Announcing Custom Graphs Public Preview](https://techcommunity.microsoft.com/blog/microsoft-security-blog/announcing-public-preview-of-custom-graphs-in-microsoft-sentinel/4507410)
- [Blog: Exposure Management Graph](https://techcommunity.microsoft.com/blog/microsoft-security-blog/microsoft-security-exposure-management-graph-unveiling-the-power/4148546)
- [Available Defender Tables](https://learn.microsoft.com/en-us/defender-xdr/advanced-hunting-schema-tables)
- [Available Azure Monitor Tables](https://learn.microsoft.com/en-us/azure/azure-monitor/reference/tables-index)
- [Available Asset Tables](https://learn.microsoft.com/en-us/azure/sentinel/datalake/enable-data-connectors)
- [Microsoft Sentinel Notebook Examples](https://learn.microsoft.com/en-us/azure/sentinel/datalake/notebook-examples)
- [Microsoft Sentinel Provider Class Reference](https://learn.microsoft.com/en-us/azure/sentinel/datalake/sentinel-provider-class-reference)
- [Custom graph overview doc](https://learn.microsoft.com/en-us/azure/sentinel/datalake/custom-graphs-overview)
- [Graph Query Language (GQL) Reference for Sentinel graph](https://learn.microsoft.com/en-us/azure/sentinel/datalake/gql-reference-for-sentinel-custom-graph)
- [Graph REST APIs for custom graphs](https://learn.microsoft.com/en-us/azure/sentinel/datalake/graph-rest-api)



## Configuration & Setup

This cell sets up the notebook environment by:
- Importing required libraries for Sentinel data access, graph building, and PySpark operations
- Configuring the workspace name, analysis time window (14 days), and logging level
- Initializing the Microsoft Sentinel provider to connect to your Sentinel Data Lake
- Displaying environment information including provider version and Spark cluster details

**Action Required**: Update `WORKSPACE_NAME` with your Microsoft Sentinel workspace name.

In [ ]:
# Import required libraries
from sentinel_lake.providers import MicrosoftSentinelProvider
from sentinel_graph.builders import GraphSpecBuilder
from pyspark.sql.functions import (
    col, concat, count, countDistinct, when, lit, expr,
    current_timestamp, avg, coalesce, first
)
from packaging import version

from importlib.metadata import version as get_version
pkg_version = get_version('MicrosoftSentinelGraphProvider')


import logging
import matplotlib.pyplot as plt
import pandas as pd

# Configuration - Update this with your workspace name
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"

# Configuration - Lookback period for Analysis
ANALYSIS_DAYS = 14

# Configuration - Logging Level (DEBUG, INFO, WARNING, ERROR)
LOGGING_LEVEL = logging.DEBUG

# Initialize Sentinel provider
sentinel_provider = MicrosoftSentinelProvider(spark)

# Set the logging level for the entire package
logging.getLogger('sentinel_graph').setLevel(LOGGING_LEVEL)  # or ERROR, WARNING, INFO, DEBUG. Default is INFO

print("="*60)
print("SCENARIO: SignInLogs Hello World.")
print("="*60)
print(f"Workspace: {WORKSPACE_NAME}")
print(f"Analysis Window: {ANALYSIS_DAYS} days")
pkg_version = get_version('MicrosoftSentinelGraphProvider')
print(f"MicrosoftSentinelGraphProvider Version: {pkg_version}")
print(f"Logging Level: {logging.getLevelName(logging.getLogger('sentinel_graph').level)}")
print(f"Spark Pool: {spark.conf.get('spark.synapse.pool.name')}")
print(f"Spark Cluster Region: {spark.conf.get('spark.cluster.region')}")
print(f"Spark Account ID: {spark.conf.get('spark.pjs.account.id')}")
print("="*60)

## Load Source Data

This cell retrieves sign-in and user data from Microsoft Sentinel:

**SigninLogs**: Loads sign-in events from the past 14 days, filtering for:
- Member users only (excluding guests)
- Non-null user IDs
- Relevant fields: user identity, resources accessed, IP addresses, and metadata

**EntraUsers**: Loads user profile information including:
- Display names, emails, and employee IDs
- Organizational data (department, country)
- Deduplicated by user ID

Both datasets are persisted in memory for efficient reuse and sample records are displayed.

In [ ]:
# ============================================================================
# Load and Filter SigninLogs Data
# ============================================================================
SigninLogs_df = (
    sentinel_provider.read_table('SigninLogs', WORKSPACE_NAME)
    .filter(
        (col("UserType").isin("Member")) &
        (col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {ANALYSIS_DAYS} DAYS")) &
        (col("UserId").isNotNull())
    )
    .select(
        # User Identity
        "Identity",
        "UserId",
        "UserPrincipalName",
        "UserDisplayName",
        
        # Resource Information
        "ResourceId",
        "ResourceDisplayName",
        "ResourceGroup",
        "AppId",
        
        # Connection Details
        "IPAddress",
        "UserAgent",
        
        # Metadata
        "CorrelationId",
        "TimeGenerated"
    )
).persist()

# ============================================================================
# Load and Filter EntraUsers Data
# ============================================================================
EntraUsers_df = (
    # Note that EntraUsers is a system table, so workspace name is not needed
    sentinel_provider.read_table('EntraUsers')
    .filter(
        (col("id").isNotNull()) &
        (col("id") != '') &
        (col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {ANALYSIS_DAYS} DAYS"))
    )
    .select(
        # User Identity
        "id",
        "displayName",
        "givenName",
        
        # Contact Information
        "mail",
        
        # Organization Details
        "department",
        "employeeId",
        "country",
        
        # Metadata
        "TimeGenerated"
    )
    .dropDuplicates(["id"])
).persist()


# ============================================================================
# Display Top 5 Records from Each DataFrame
# ============================================================================

print("\n📊 SigninLogs Sample:")
SigninLogs_df.show(5, truncate=False)

print("\n👥 EntraUsers Sample:")
EntraUsers_df.show(5, truncate=False)

## Create Graph-ready DataFrames

This cell transforms the raw data into graph-ready structures:

**Node Types:**
- **User Nodes**: Created from EntraUsers with all profile attributes
- **Department Nodes**: Unique departments extracted from user data
- **Application Nodes**: Unique applications from sign-in logs (identified by AppId and ResourceId)

**Edge Types:**
- **BelongsTo Edges**: Connect users to their departments
- **CommunicatedWith Edges**: Connect users to applications with:
  - Connection counts (number of sign-ins)
  - First time seen timestamps
  - IP address and user agent information

Sample data from each DataFrame is displayed for verification.

In [ ]:
# ============================================================================
# Create Data Frames for each Node and each Edge type
# ============================================================================

# ----------------------------------------------------------------------------
# NODE TYPE: User
# Create User nodes from EntraUsers data
# ----------------------------------------------------------------------------
UserNodes_df = (
    EntraUsers_df
    .withColumn("nodeType", lit("User"))
)

# ----------------------------------------------------------------------------
# NODE TYPE: Department
# Create Department nodes from unique departments in EntraUsers
# ----------------------------------------------------------------------------
DepartmentNodes_df = (
    EntraUsers_df
    .selectExpr("Department as Org")
    .distinct()
    .withColumn("nodeType", lit("Department"))
)

# ----------------------------------------------------------------------------
# NODE TYPE: Application
# Create Application nodes from unique apps in SigninLogs
# ----------------------------------------------------------------------------
ApplicationNodes_df = (
    SigninLogs_df
    .selectExpr(
        "ResourceId",
        "AppId",
        "ResourceDisplayName as AppName"
    )
    .dropDuplicates(["ResourceId", "AppId"])
    .withColumn("nodeType", lit("App"))
)

# ----------------------------------------------------------------------------
# EDGE TYPE: BelongsTo
# Create edges connecting Users to Departments
# ----------------------------------------------------------------------------
BelongsToEdges_df = (
    EntraUsers_df
    .withColumn("edgeType", lit("BelongsTo"))
    .withColumn("UserId_BT", col("id"))
)

# ----------------------------------------------------------------------------
# EDGE TYPE: CommunicatedWith
# Create edges connecting Users to Applications with communication metrics
# ----------------------------------------------------------------------------
CommunicatedWithEdges_df = (
    SigninLogs_df
    .groupBy(
        col("UserId"),
        col("IPAddress"),
        col("AppId"),
        col("UserAgent"),
        col("UserId").alias("UserId_dest"),
        lit("communicatedWith").alias("edgeType")
    )
    .agg(
        count("*").alias("count"),
        first("TimeGenerated").alias("FirstTimeSeen")
    )
)

# ============================================================================
# Display Sample Data from Each DataFrame
# ============================================================================

print("\n👤 User Nodes:")
EntraUsers_df.show(5, truncate=False)

print("\n🏢 Department Nodes:")
DepartmentNodes_df.show(5, truncate=False)

print("\n📱 Application Nodes:")
ApplicationNodes_df.show(5, truncate=False)

print("\n🔗 BelongsTo Edges (User → Department):")
BelongsToEdges_df.show(5, truncate=False)

print("\n🔗 CommunicatedWith Edges (User → Application):")
CommunicatedWithEdges_df.show(5, truncate=False)

## Define Graph Structure

This cell uses the GraphSpecBuilder to define the complete graph structure:

**Nodes Configuration:**
- Users node with identity columns (id, displayName, mail, department, etc.)
- Applications node with app identifiers (AppId, ResourceId, AppName)
- Department node with organization name

**Edges Configuration:**
- BelongsTo relationship linking Users → Departments
- CommunicatedWith relationship linking Users → Applications with communication metrics

The builder auto-generates system fields (sys_id, sys_label) and prepares the schema for graph creation.


In [ ]:
builder = (GraphSpecBuilder.start()

    .add_node("Users") \
        .from_dataframe(UserNodes_df.df) \
        .with_columns("country", "department", "displayName", "employeeId", "givenName", "id", "mail", "TimeGenerated", "nodeType", key="id", display="id")

    .add_node("Applications") \
        .from_dataframe(ApplicationNodes_df.df) \
        .with_columns("ResourceId", "AppId", "AppName", "nodeType", key ="AppId", display="AppId")

     .add_node("Department") \
        .from_dataframe(DepartmentNodes_df.df) \
        .with_columns("Org", "nodeType", key="Org", display="Org")

    .add_edge("BelongsTo") \
        .from_dataframe(BelongsToEdges_df.df) \
        .source(id_column="UserId_BT", node_type="Users") \
        .target(id_column="department", node_type="Department") \
        .with_columns("edgeType","TimeGenerated", key ="edgeType", display="edgeType")
    
    .add_edge("communicatedWith") \
        .from_dataframe(CommunicatedWithEdges_df) \
        .source(id_column="UserId", node_type="Users") \
        .target(id_column="AppId", node_type="Applications") \
        .with_columns("edgeType","count", "FirstTimeSeen",
                  key="edgeType", display="edgeType")
   
).done()


## Build and Publish Graph

This cell executes the graph build process:

1. **Data Transformation**: Converts DataFrames to unified node and edge tables
2. **Upload to Sentinel**: Writes graph data to Sentinel's graph storage (Delta format)
3. **Graph Registration**: Creates a graph instance via Sentinel API
4. **Polling**: Monitors graph creation status until it's ready for queries (typically ~2 minutes)

The graph is persisted and accessible for querying through GQL (Graph Query Language).

**Result**: Graph instance status should show `PUBLISHED` when complete.


In [ ]:
build_result = builder.build_graph_with_data()

print(f"Status: {build_result.get('status')}")

## Identify Most Connected Users Using Centrality Analysis

This cell runs a **Betweenness Centrality** query to find users with the most connections to applications:

**Query Parameters:**
- Source nodes: Users
- Target nodes: Applications  
- Edges: communicatedWith relationships
- Analysis: Counts the number of paths through each user

**Output**: A ranked list of users by their "NumberOfPaths" (centrality score), indicating which users interact with the most applications. Users with high centrality may be:
- Power users or administrators
- Potential security risks if compromised
- Good candidates for behavior baselining

The top user's email and department are captured for further analysis.


In [ ]:
from sentinel_graph.builders.query_input import CentralityQueryInput
from pyspark.sql.functions import col, from_json, schema_of_json

result = builder.centrality(CentralityQueryInput(
    participating_source_node_labels=["Users"],
    participating_target_node_labels=["Applications"],
    participating_edge_labels=["communicatedWith"],
    is_directional=True,
#    threshold=5
))

result_df = result.to_dataframe()

# Parse and sort, filtering out blank emails
sample_json = result_df.select("Node").first()[0]
schema = schema_of_json(sample_json)

mail_sorted_df = (
    result_df
    .withColumn("parsed", from_json(col("Node"), schema))
    .select(
        col("parsed.mail").alias("mail"),
        col("parsed.NumberOfPaths").alias("NumberOfPaths"),
        col("parsed.department").alias("department")
    )
    .filter(col("mail").isNotNull() & (col("mail") != ""))  # Filter blanks
    .orderBy(col("NumberOfPaths").desc())
)

top_user_email = mail_sorted_df.first().mail
top_user_department = mail_sorted_df.first().department

mail_sorted_df.show(10, truncate=False)

## Visualize All Relationships for the Most Central User

This cell executes a GQL query to retrieve all edges connected to the most central user:

**Query**: `MATCH (n)-[e]->(s) WHERE n.mail = '{top_user_email}' RETURN *`

This returns:
- The user node (n)
- All outbound relationships (e) - both BelongsTo and communicatedWith edges
- All connected nodes (s) - departments and applications

**Visualization**: Results are displayed as an interactive graph showing:
- Which applications the user accessed
- How many times they accessed each application
- The user's department affiliation
- Temporal patterns (FirstTimeSeen timestamps)


In [ ]:
query1 = f"MATCH (n)-[e]->(s) WHERE n.mail = '{top_user_email}' RETURN *"

builder.query(query1).show()

## Analyze All Users in the Top User's Department

This cell expands the analysis to the entire department:

**Query**: `MATCH (n)-[e]->(s) WHERE n.department = '{top_user_department}' RETURN *`

This reveals:
- All users in the same department as the most central user
- All applications accessed by department members
- Communication patterns across the entire department
- Potential anomalies (users with unusual access patterns compared to peers)

**Use Cases:**
- Identify department-wide application dependencies
- Detect insider threat patterns
- Understand normal department behavior for baseline creation
- Find shared access patterns or shadow IT usage

In [ ]:
query2 = f"MATCH (n:Users)-[e]->(s) WHERE n.department = '{top_user_department}' RETURN * LIMIT 50"

builder.query(query2).show()

## All user-app communications for users in top user's department

In [ ]:
query3 = f"MATCH (n)-[e:communicatedWith]->(a), (n)-[b:BelongsTo]->(d) WHERE d.Org IN ['{top_user_department}'] RETURN * LIMIT 100"


builder.query(query3).show()